In [45]:
try:
    from google.colab import drive  # type: ignore
except ImportError:
    pass
else:
    drive.mount("/content/drive")
    !cp /content/drive/MyDrive/BuildGPT/Part5/name.txt /content/ 

In [46]:
with open("name.txt", "r") as file:
    words = file.read().splitlines()
print(len(words))
print(max(len(w) for w in words))
print(words[:8])

32033
15
['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']


In [47]:
import torch
import torch.nn.functional as F
import random
from typing import Any

In [48]:
chars: list[str] = sorted(set("".join(words)))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi["."] = 0
itos = {i: s for s, i in stoi.items()}

vocab_size = len(stoi)

In [49]:
random.seed(42)
random.shuffle(words)

In [50]:
block_size = 3


def build_data_set(words: list[str]):
    X, Y = [], []
    for word in words:
        context = [0] * block_size
        for char in word:
            ix = stoi[char]
            X.append(context)  # X 存储 context 字符索引
            Y.append(ix)  # Y 存储预测 的字符索引
            context = context[1:] + [ix]  # 每次都把上下文的第一个字符索引去掉

    return torch.tensor(X), torch.tensor(Y)


X, Y = build_data_set(words)
print(f"X shape: {X.shape}, Y shape:{Y.shape}")

n_words = len(words)
n1 = int(0.8 * n_words)
n2 = int(0.9 * n_words)

X_train, Y_train = build_data_set(words[:n1])
X_dev, Y_dev = build_data_set(words[n1:n2])
X_test, Y_test = build_data_set(words[n2:])

X shape: torch.Size([196113, 3]), Y shape:torch.Size([196113])


In [51]:
class Linear:
    # fan_out: 神经元的数量
    # fan_in:单个神经元可以接受多少个参数
    def __init__(self, fan_in: int, fan_out: int, bias=True):
        self.weight = torch.randn(fan_in, fan_out) * fan_in ** (-0.5)  # kaiming init
        self.bias = torch.zeros(fan_out) if bias else None

    def __call__(self, x: torch.Tensor):
        self.out = x @ self.weight
        if self.bias is not None:
            self.out += self.bias
        return self.out

    def parameters(self):
        return [self.weight] + ([] if self.bias is None else [self.bias])

In [52]:
class BatchNorm1d:
    def __init__(self, dim, eps=1e-5, momentum=0.1) -> None:
        self.eps = eps
        self.momentum = momentum
        self.training = True

        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)

        # batch 均值和方差
        self.x_mean = torch.zeros(dim)
        self.x_var = torch.zeros(dim)

        # 全局均值和方差
        self.running_mean = torch.zeros(dim)
        self.running_var = torch.zeros(dim)

    def __call__(self, x: torch.Tensor) -> Any:
        if x.dim() == 2:
            dims = (0,)
        elif x.dim() == 3:
            dims = (0, 1)
        else:
            dims = (0,)

        if self.training:
            x_mean = x.mean(dim=dims, keepdim=True)
            x_var = x.var(dim=dims, keepdim=True)
        else:
            x_mean = self.running_mean
            x_var = self.running_var

        x_hat = (x - x_mean) / ((x_var + self.eps) ** (0.5))  # batch normalization
        self.out = self.gamma * x_hat + self.beta

        # train 的过程同时计算全局 mean, var
        if self.training:
            with torch.no_grad():
                self.running_mean = (
                    1 - self.momentum
                ) * self.running_mean + self.momentum * x_mean
                self.running_var = (
                    1 - self.momentum
                ) * self.running_var + self.momentum * x_var

        return self.out

    def parameters(self):
        return [self.gamma, self.beta]

**三维 Tensor 的平均**

```python
T.type : torch.Tensor
T.shape = (3, 5, 4) 
```

在这个矩阵中，一共有 3 × 5 = 15 个格子。

每一个格子里的元素 $v_{i,j}$ 不是一个单独的数值，而是一个包含 4 个分量的向量：$$v_{i,j} = [c_1, c_2, c_3, c_4]$$

其中 $i$ 表示行号（1 到 3），$j$ 表示列号（1 到 5）。

执行 mean(dim=(0, 1))

15 个c_1 求平均, 15 个c_2 求平均, 15 个c_3 求平均, 15 个c_4 求平均, 最终的 Tensor.shape = (1, 1, 4)


In [53]:
class Tanh:
    def __call__(self, x):
        self.out = torch.tanh(x)
        return self.out

    def parameters(self):
        return []

In [54]:
class Embedding:
    # num_embbding, 要嵌入到多少个元素中
    # embedding_dim, 每个元素需要多少个特征
    def __init__(self, num_embbding, embedding_dim) -> None:
        self.weight = torch.randn(num_embbding, embedding_dim)

    def __call__(self, IX):  # IX 表示这里嵌入的是 indices, 区别与其它 x
        self.out = self.weight[IX]
        return self.out

    def parameters(self):
        return [self.weight]

In [55]:
class Flatten:
    def __call__(self, x):
        self.out = x.view(x.shape[0], -1)
        return self.out

    def parameters(self):
        return []

In [56]:
# class FlattenConsecutive:
#     def __init__(self, n) -> None:
#         self.n = n

#     def __call__(self, x:torch.Tensor):
#         B = x.shape[0] #batch
#         T = x.shape[1] # Time
#         C = x.shape[2] # Channel
#         x.view(B, T // self.n, C * self.n) # // 表示整除, 向下取整

#         if x.shape[1] == 1: # 如果该维度被 flatten 到 1 了, 那就 取消该维度, 也就是视为 2 维
#             x.squeeze(1)

#         self.out = x

#         return x

#     def parameters(self):
#         return []

**Flatten 的本质**

假设一个输入张量，形状为 $(32, 8, 10)$, $32$ 个句子, 每个句子有 $8$ 个单词, 每个单词用长度为 $10$ 。

使用 FlattenConsecutive 并且设定每次融合相邻的 $2$ 个单词：它会把第 1、2 个单词的向量拼接，第 3、4 个拼接，依此类推。

单词数维度被压缩一半（$8 \div 2 = 4$），而最后的向量维度(描述单个单词的向量的分量)膨胀一倍（$10 \times 2 = 20$）。

结果形状：$(32, 4, 20)$。

网络像“锦标赛”或者“树状图”一样，一层一层地把低级特征（单字）两两组合成高级特征（词组）

In [57]:
from typing import Any


class Sequential:
    def __init__(self, layers: list[Any]) -> None:
        self.layers = layers

    def __call__(self, x: torch.Tensor) -> Any:
        for layer in self.layers:
            x = layer(x)

        self.out = x

        return self.out

    def parameters(self) -> list[torch.Tensor]:
        return [p for layer in self.layers for p in layer.parameters()]  # 列表推导

In [58]:
torch.manual_seed(42)

**最初的形式, 下面会在此基础上优化**
```python
# 初始化
embdding_dim = 10
n_hidden = 200


layers = [
    Embedding(vocab_size, embdding_dim),
    Flatten(),
    Linear(embdding_dim * block_size, n_hidden, bias=False),
    BatchNorm1d(n_hidden), # x线性层的列, 会自动行扩展
    Tanh(), Linear(n_hidden, vocab_size)]

with torch.no_grad():
    # 模型初始状态下应该是无知的,平等的看待任何元素
    # 因而人为的调整 logits 为 0, 这里仅接近 0, 没有设置为绝对 0
    layers[-1].weight *= 0.1

parameters:list[torch.Tensor] = [p for layer in layers for p in layer.parameters()]
print(sum(p.nelement() for p in parameters))

for p in parameters:
    p.requires_grad = True
```

___

```python
max_steps = 20000
batch_size = 32

lossi = []
for i in range(max_steps):
    # minibatch construct
    ix = torch.randint(0, X_train.shape[0], (batch_size,))
    Xb, Yb = X_train[ix], Y_train[ix] # train 嵌入到 ix
    
    # forward pass
    x= Xb 
    for layer in layers:
        x=layer(x)
    # x 即 logits
    loss = F.cross_entropy(x, Yb)
    
    lr = 0.1 if i < 150000 else 0.01

    for p in parameters:
        if p.grad is not None:
            p.data += -lr * p.grad

    print(f"{i:7d} / {max_steps:7d}: {loss.item():.4f}")
    
    lossi.append(loss.log10().item())
    
    break
```

In [59]:
# 初始化
embdding_dim = 10
n_hidden = 200


model = Sequential(
    [
        Embedding(vocab_size, embdding_dim),
        Flatten(),
        Linear(embdding_dim * block_size, n_hidden, bias=False),
        BatchNorm1d(n_hidden),  # x线性层的列, 会自动行扩展
        Tanh(),
        Linear(n_hidden, vocab_size),
    ]
)

with torch.no_grad():
    # 模型初始状态下应该是无知的,平等的看待任何元素
    # 因而人为的调整 logits 为 0, 这里仅接近 0, 没有设置为绝对 0
    model.layers[-1].weight *= 0.1

parameters = model.parameters()
print(sum(p.nelement() for p in parameters))

for p in parameters:
    p.requires_grad = True

12097


In [60]:
######################## train ####################################
max_steps = 20000
batch_size = 32

lossi = []
for i in range(max_steps):
    # minibatch construct
    ix = torch.randint(0, X_train.shape[0], (batch_size,))
    Xb, Yb = X_train[ix], Y_train[ix]  # train 嵌入到 ix

    # forward pass
    logits = model(Xb)
    loss = F.cross_entropy(logits, Yb)

    lr = 0.1 if i < 150000 else 0.01

    for p in parameters:
        if p.grad is not None:
            p.data += -lr * p.grad

    print(f"{i:7d} / {max_steps:7d}: {loss.item():.4f}")

    lossi.append(loss.log10().item())

    break

      0 /   20000: 3.2907


In [61]:
######################## evaluate the loss ####################################
@torch.no_grad()
def split_loss(split: str):
    x, y = {
        "train": (X_train, Y_train),
        "dev": (X_dev, Y_dev),
        "test": (X_test, Y_test),
    }[split] # 从 map 中取出
    
    logits = model(x)
    loss = F.cross_entropy(logits, y)
    
    print(f"{split}: {loss.item():.4f}")

split_loss("train")
split_loss("dev")

train: 3.3018
dev: 3.3023


In [ ]:
######################## sample from the model ####################################
for layer in model.layers:
    layer.training = False # 解释见下
    

for _ in range(20):
    context = [0] * block_size
    out=[]
    while(True):
        logits = model(torch.tensor([context]))
        probs = F.softmax(logits, dim=1)
        ix = torch.multinomial(probs, num_samples=1).item() # 采样结果是下标, 优先采样概率(权重)高的元素
        context = context[1:] + [ix]
        out.append(ix)
        
        if ix == 0: # 预测的是 "."
            break
    print(''.join(itos[i] for i in out))


nlsmiwfnne.
ymoovjowxrsdsjfmfjfmsgaukcuza.
yz.
ss.
vcqaejgwn.
j.
mmvoknmcxwbbgrtzxgfqzulywptmibgneghwerpgkkvasrodcdzndifbcumklepzzxuypalyuluvntoeibadmzkntcbyutmhbfvskusgtvfh.
qrbuwwtichpf.
ou.
wdudntdjzbgtediamkfyvvrilwcwaifqxomwjmujgyrlnqd.
xnintfhzikgnnoihmqscnfvbulieullncptvihjsliatzhefobivxmfmdsl.
oph.
czikrbdgzajgmunlcklckqvxcflewpvnxofdcjvyotrwalaqxboftu.
pmhxxgsjtoucbah.
wqcedwhnlqmfgkxjbzzf.
bx.
twcuyaqucypgdstctolmygxvzutwdhrrjzscfrphfrava.
laqxeyzcahzsdlgnxyz.
ogefyigjgbs.
yggquvtmrgjcnohwihtflyougan.


**为什么要 layer.training = False**

`class BatchNorm1d` 中，默认状态是 `self.training = True`

意味着哪怕只有 1 个样本，它也要强行去计算当前 Batch 的方差。

统计学中计算无偏样本方差（Bessel's correction）的公式，需要除以 $N - 1$：

$$s^2 = \frac{1}{N-1} \sum_{i=1}^{N} (x_i - \bar{x})^2$$

当 $N = 1$（只有一个样本）时，分母变成了 $0$！PyTorch 计算单样本方差会直接返回 nan。

归一化代码 `(x - x_mean) / sqrt(x_var + eps)` 因为分母含有 nan，输出全变成了 nan。